# EduInsight — Notebook 2: Data Preparation

## Objective

Create one leakage-aware modelling row per eligible student using only
information available from course day 0 through day 30.

## 1. Reproducible project configuration

This notebook intentionally repeats its configuration and imports. Jupyter
notebooks must be executable independently and in order from a clean kernel.

In [1]:
from pathlib import Path
import sys


def locate_project_root(
    start: Path | None = None,
) -> Path:
    current = (
        start or Path.cwd()
    ).resolve()

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            (candidate / "src").is_dir()
            and (
                candidate / "notebooks"
            ).is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Open VS Code from the repository "
        "folder and run the notebook again."
    )


PROJECT_ROOT = locate_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)


SELECTED_MODULE = "BBB"
SELECTED_PRESENTATION = "2013J"
CUTOFF_DAY = 30


print(
    f"Project root: "
    f"{PROJECT_ROOT}"
)

Project root: D:\Work\GIT\eduinsight-student-success


## 2. Imports and source-table loading

In [2]:
import pandas as pd

from src.data_preparation import (
    load_csv,
    validate_raw_data_files,
)

from src.feature_engineering import (
    build_early_assessment_features,
    build_early_vle_features,
    build_student_feature_table,
    select_eligible_students,
)


pd.set_option(
    "display.max_columns",
    None,
)


validate_raw_data_files(
    RAW_DATA_DIR
)


student_info = load_csv(
    RAW_DATA_DIR,
    "studentInfo.csv",
)

student_registration = load_csv(
    RAW_DATA_DIR,
    "studentRegistration.csv",
)

assessments = load_csv(
    RAW_DATA_DIR,
    "assessments.csv",
)

student_assessments = load_csv(
    RAW_DATA_DIR,
    "studentAssessment.csv",
)


print(
    "Small source tables loaded successfully."
)

Small source tables loaded successfully.


## 3. Define the observation window

The model will make a demonstration estimate at day 30. Consequently, all
behavioural features must use dates no later than day 30. Using later data
would create **data leakage**.

In [3]:
print(
    f"Selected presentation: "
    f"{SELECTED_MODULE} "
    f"{SELECTED_PRESENTATION}"
)

print(
    f"Observation window: "
    f"course days 0 through "
    f"{CUTOFF_DAY}"
)

Selected presentation: BBB 2013J
Observation window: course days 0 through 30


## 4. Select students eligible for a day-30 prediction

In [4]:
eligible_students = (
    select_eligible_students(
        student_info,
        student_registration,
        module=SELECTED_MODULE,
        presentation=(
            SELECTED_PRESENTATION
        ),
        cutoff_day=CUTOFF_DAY,
    )
)


print(
    f"Eligible students: "
    f"{len(eligible_students):,}"
)


eligible_students.head()

Eligible students: 1,825


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration
0,BBB,2013J,23798,M,Wales,A Level or Equivalent,50-60%,0-35,0,60,N,Distinction,-27.0
1,BBB,2013J,27759,M,North Western Region,Lower Than A Level,40-50%,35-55,0,120,Y,Fail,-43.0
2,BBB,2013J,30091,F,South West Region,A Level or Equivalent,10-20,0-35,0,60,Y,Pass,-145.0
3,BBB,2013J,31014,F,West Midlands Region,Lower Than A Level,80-90%,35-55,0,120,N,Withdrawn,-43.0
4,BBB,2013J,31849,F,South West Region,Lower Than A Level,60-70%,35-55,0,120,N,Pass,-128.0


Students who withdrew on or before day 30 are excluded because their
withdrawal is already known at prediction time. Students who withdraw after
day 30 remain eligible because that later outcome is part of the target.

## 5. Build early VLE engagement features

In [5]:
vle_features = (
    build_early_vle_features(
        RAW_DATA_DIR
        / "studentVle.csv",
        module=SELECTED_MODULE,
        presentation=(
            SELECTED_PRESENTATION
        ),
        cutoff_day=CUTOFF_DAY,
        chunksize=500_000,
    )
)


print(
    "Students with early "
    "VLE activity: "
    f"{len(vle_features):,}"
)


vle_features.head()

Students with early VLE activity: 1,771


,code_module,code_presentation,id_student,total_clicks_30,active_days_30,unique_resources_30,avg_clicks_per_active_day_30
0,BBB,2013J,23798,146,11,15,13.272727
1,BBB,2013J,27759,123,12,21,10.250000
2,BBB,2013J,30091,162,8,22,20.250000
3,BBB,2013J,31014,311,22,10,14.136364
4,BBB,2013J,31849,414,17,23,24.352941


The large VLE file is read in chunks. Only the selected presentation and
day-0-to-day-30 rows are retained before student-level aggregation.

## 6. Build early assessment features

In [6]:
assessment_features = (
    build_early_assessment_features(
        assessments,
        student_assessments,
        module=SELECTED_MODULE,
        presentation=(
            SELECTED_PRESENTATION
        ),
        cutoff_day=CUTOFF_DAY,
    )
)


print(
    "Students with at least one "
    "assessment submitted by day 30: "
    f"{len(assessment_features):,}"
)


assessment_features.head()

Students with at least one assessment submitted by day 30: 1,607


,code_module,code_presentation,id_student,assessments_submitted_30,avg_score_30,late_submissions_30,has_early_assessment
0,BBB,2013J,23798,1,90.0,0,1
1,BBB,2013J,27759,1,61.0,0,1
2,BBB,2013J,30091,1,80.0,1,1
3,BBB,2013J,31014,1,85.0,0,1
4,BBB,2013J,31849,1,81.0,0,1


## 7. Assemble one row per student

In [7]:
model_data = (
    build_student_feature_table(
        eligible_students,
        vle_features,
        assessment_features,
    )
)


print(
    f"Prepared rows: "
    f"{len(model_data):,}"
)

print(
    f"Prepared columns: "
    f"{model_data.shape[1]}"
)


model_data.head()

Prepared rows: 1,825
Prepared columns: 22


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,total_clicks_30,active_days_30,unique_resources_30,avg_clicks_per_active_day_30,assessments_submitted_30,avg_score_30,late_submissions_30,has_early_assessment,support_needed
0,BBB,2013J,23798,M,Wales,A Level or Equivalent,50-60%,0-35,0,60,N,Distinction,-27.0,146,11,15,13.272727,1,90.0,0,1,0
1,BBB,2013J,27759,M,North Western Region,Lower Than A Level,40-50%,35-55,0,120,Y,Fail,-43.0,123,12,21,10.250000,1,61.0,0,1,1
2,BBB,2013J,30091,F,South West Region,A Level or Equivalent,10-20,0-35,0,60,Y,Pass,-145.0,162,8,22,20.250000,1,80.0,1,1,0
3,BBB,2013J,31014,F,West Midlands Region,Lower Than A Level,80-90%,35-55,0,120,N,Withdrawn,-43.0,311,22,10,14.136364,1,85.0,0,1,1
4,BBB,2013J,31849,F,South West Region,Lower Than A Level,60-70%,35-55,0,120,N,Pass,-128.0,414,17,23,24.352941,1,81.0,0,1,0


## 8. Quality checks

In [8]:
student_key = [
    "code_module",
    "code_presentation",
    "id_student",
]


assert not model_data.duplicated(
    subset=student_key
).any()


assert model_data[
    "support_needed"
].isin([0, 1]).all()


assert (
    model_data[
        "total_clicks_30"
    ] >= 0
).all()


assert (
    model_data[
        "active_days_30"
    ] >= 0
).all()


assert (
    "date_unregistration"
    not in model_data.columns
)


print(
    "All preparation checks passed."
)

All preparation checks passed.


## 9. Review the target and missing values

In [9]:
target_summary = (
    model_data[
        "support_needed"
    ]
    .value_counts()
    .rename(
        index={
            0: "No support flag",
            1: "Support flag",
        }
    )
    .to_frame("students")
)


target_summary[
    "percentage"
] = (
    target_summary["students"]
    / len(model_data)
    * 100
).round(2)


target_summary

,students,percentage
support_needed,,
No support flag,1071,58.68
Support flag,754,41.32


In [10]:
(
    model_data
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
    .to_frame(
        "missing_values"
    )
)

,missing_values
avg_score_30,225
imd_band,14
id_student,0
gender,0
code_module,0
code_presentation,0
highest_education,0
region,0
num_of_prev_attempts,0
age_band,0


A missing `avg_score_30` is retained when no early score exists. The model
pipeline will impute it and add a missingness indicator during training.

## 10. Save the processed feature table

In [11]:
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


output_path = (
    PROCESSED_DATA_DIR
    / "student_features.csv"
)


model_data.to_csv(
    output_path,
    index=False,
)


print(
    f"Saved: {output_path}"
)

Saved: D:\Work\GIT\eduinsight-student-success\data\processed\student_features.csv


## Completion check

Notebook 2 is complete when it runs from a clean kernel, all assertions pass,
and `data/processed/student_features.csv` is created.